<a href="https://colab.research.google.com/github/avindumihisara0229-code/ErgoSense/blob/Tharusha/DSGP_SVM_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install librosa soundfile

In [5]:
import os
import numpy as np
import pandas as pd
import librosa

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [7]:
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/archive"

In [8]:
def get_stress_label(filename):
    emotion_code = int(filename.split("-")[2])

    if emotion_code in [5, 6]:
        return 1
    elif emotion_code in [1, 2, 3]:
        return 0
    else:
        return None

In [2]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=3, offset=0.5)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs, axis=1)

    pitches, _ = librosa.piptrack(y=y, sr=sr)
    pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

    energy = np.mean(librosa.feature.rms(y=y))

    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    return np.hstack([mfccs_mean, pitch_mean, energy, tempo])

In [10]:
features = []
labels = []

for actor_folder in os.listdir(DATASET_PATH):
    actor_path = os.path.join(DATASET_PATH, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                label = get_stress_label(file)

                if label is not None:
                    file_path = os.path.join(actor_path, file)
                    feature_vector = extract_features(file_path)

                    features.append(feature_vector)
                    labels.append(label)

In [11]:
feature_columns = [f"mfcc_{i}" for i in range(1, 14)]
feature_columns += ["pitch", "energy", "tempo"]

df = pd.DataFrame(features, columns=feature_columns)
df["stress"] = labels

df.head()

,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13,pitch,energy,tempo,stress
0,-603.358459,55.877415,-11.653138,5.715710,-7.732581,-6.145287,-7.697657,-12.779858,-7.422775,-3.974831,-6.918380,-4.023993,-10.244540,1652.461548,0.005791,172.265625,0
1,-454.529999,23.917503,-30.110622,-4.554773,-13.989171,-12.565503,-13.440819,-18.867109,-6.790040,-3.687633,-12.427094,-3.805950,-7.189543,1865.881104,0.018929,198.768029,1
2,-591.921143,62.216999,-15.930927,0.886865,-8.853312,-7.244374,-10.352101,-15.372656,-8.205089,-4.805441,-8.849456,-7.176614,-12.285439,1768.306030,0.004814,112.347147,0
3,-526.763916,52.980293,-13.038193,7.258233,-7.994298,-5.933784,-4.651824,-12.674568,-5.665134,-2.909221,-3.826861,-3.017372,-6.013659,1737.567993,0.008449,129.199219,1
4,-614.896851,65.422890,-5.475077,14.217050,-6.454205,-2.532207,-4.585760,-9.615055,-3.781765,-2.738353,-4.148098,-5.384352,-5.899535,1664.050903,0.004210,198.768029,0


In [12]:
df["stress"].value_counts()

,count
stress,
0,480
1,384


In [13]:
df.to_csv("ravdess_stress_features.csv", index=False)
print("Feature extraction complete. CSV saved!")

Feature extraction complete. CSV saved!


In [14]:
from google.colab import files
files.download("ravdess_stress_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>